# La velocidad de C1EÑ en GPU: jit · nativo · T4, contra NumPy y CuPy

Este cuaderno **mide** lo que hasta ahora eran estimaciones. Corre el mismo programa de Ñ (`c1en/velocidad-c1en.esl`) en los tres dispositivos de C1EÑ —`jit` (JavaScript), `nativo` (C con gcc y OpenMP en los vCPU) y `gpu` (CUDA C con NVRTC en una T4)— y lo pone al lado de NumPy (CPU) y CuPy (GPU) haciendo lo mismo.

Tres familias de pruebas:

| familia | qué es | ejemplos |
|---|---|---|
| **nucleos** | funciones de Ñ lanzadas con `paralelo()`: un hilo por elemento | saxpy, π por Monte Carlo, mandelbrot, n-cuerpos, histograma con `sumarAtomico` |
| **nen** | operaciones de NEÑ en volumen sobre arreglos; pasan por C1EÑ desde 32.768 elementos. Desde motor-exp N2 los arreglos se quedan en la GPU entre operaciones: `senoA` mide con los datos ya allí (como `cupy`), `senoA+copia` subiendo y bajando (como CuPy con copia) y `cadena` es `suma(expA(senoA(x)) * 2 + 1)` entera | `senoA`, `expA`, `senoA+copia`, `cadena`, `softmax`, `sumaEje`, `por` (matrices), `fourier` |
| **arit** | aritmética de arreglos y suma: desde motor-exp N2 también van por C1EÑ si el arreglo es grande | `x * 2 + 1`, `suma` |
| **ecosim** | ECOSIM-G v3.1 entero: ms por tick y por mundo | 100 · 1.000 mundos |

Lo que sale: `c1en-velocidad.csv` (una fila por prueba, tamaño, motor y dispositivo, con ms, rendimiento y la huella del resultado), `c1en-velocidad-resumen.csv` (la tabla ancha) y las gráficas. **La huella** es una suma del resultado: en los núcleos sin funciones trascendentes tiene que salir **idéntica** en jit, nativo y gpu (la promesa de C1EÑ); la celda 8 lo comprueba.

**Antes de empezar:** *Settings → Accelerator → GPU T4 x2*. Para la opción B (clonar de GitHub), *Settings → Internet → On*. Con la escala `normal` tarda unos 15–25 minutos (casi todo es el `jit`, que es el lento a propósito: es la base contra la que se mide).

In [ ]:
# 1 · Node 22 (Kaggle trae uno viejo o ninguno) — se instala en /kaggle/working sin tocar el sistema
import os, subprocess, shutil, re, glob, time, json
def version_node():
    try: return subprocess.run(['node', '--version'], capture_output=True, text=True).stdout.strip()
    except FileNotFoundError: return ''
v = version_node()
if not v or int(re.match(r'v(\d+)', v).group(1)) < 18:
    NODE = 'v22.12.0'
    subprocess.run(f'curl -fsSL https://nodejs.org/dist/{NODE}/node-{NODE}-linux-x64.tar.xz | tar -xJ -C /kaggle/working', shell=True, check=True)
    os.environ['PATH'] = f'/kaggle/working/node-{NODE}-linux-x64/bin:' + os.environ['PATH']
print('node', version_node())

In [ ]:
# 2 · El proyecto. Opción A: un dataset de Kaggle con la carpeta del proyecto (la que tiene motor/ dentro).
#     Opción B: git clone del repositorio. Se usa A si se encuentra; si no, B.
PROYECTO = '/kaggle/working/n'
candidatos = [os.path.dirname(os.path.dirname(p)) for p in glob.glob('/kaggle/input/**/motor/rutas.js', recursive=True)]
if os.path.exists(PROYECTO):
    print('ya estaba en', PROYECTO)
elif candidatos:
    shutil.copytree(candidatos[0], PROYECTO, ignore=shutil.ignore_patterns('node_modules'))
    print('opción A: copiado desde', candidatos[0])
else:
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/diegovelandiabarajas1-lang/Espanol-Like', PROYECTO], check=True)
    print('opción B: clonado en', PROYECTO)
os.chdir(PROYECTO)
if not os.path.exists('c1en/velocidad-c1en.esl'):
    commit = subprocess.run(['git', 'log', '-1', '--format=%h %ad %s', '--date=short'], capture_output=True, text=True).stdout.strip()
    raise SystemExit('falta c1en/velocidad-c1en.esl (fase N1). ' +
                     (f'El repositorio clonado está en «{commit}»: haz git push desde tu PC y vuelve a correr. ' if commit else '') +
                     'O sube la carpeta del proyecto como dataset (opción A).')
assert os.path.exists('motor/src/14-c1en.js'), 'este proyecto no trae C1EÑ'
print(subprocess.run(['git', 'log', '-1', '--format=%h %s'], capture_output=True, text=True).stdout.strip())

## Qué motor y qué escala

- `MOTORES`: `motor-exp` es donde está la fase N1 (multiplicar matrices por bloques de 4×8 en la CPU, reducciones sin copiar, el tipo `grande`…). Añade `'motor'` para medir también el LTS y ver la diferencia.
- `ESCALA`: `chica` (comprobar que todo corre, 1–2 min), `normal` (la de la T4) o `grande` (arreglos de hasta 40 millones y matrices de 4.096: deja que la GPU se luzca, pero el `jit` se salta casi todo por el presupuesto).
- `PRESUPUESTO_MS`: si una prueba tarda más que esto, se repite menos y sus tamaños mayores se saltan en ese dispositivo (el `jit` con mil millones de muestras de π tardaría minutos).

In [ ]:
MOTORES = ['motor-exp']            # añade 'motor' para comparar con el LTS
DISPOSITIVOS = ['jit', 'nativo', 'gpu']
ESCALA = 'normal'                  # 'chica' · 'normal' · 'grande'
REPETICIONES = 3                   # se queda la mejor
PRESUPUESTO_MS = 20000
GPU = 0                            # la T4 que se usa (C1EN_GPU); la otra queda libre
TRABAJO = '/kaggle/working'

In [ ]:
# 3 · koffi (el FFI de Node que habla con libcuda y libnvrtc), los bundles y las GPU
!cd /kaggle/working/n && npm install koffi --no-audit --no-fund --loglevel=error
for m in MOTORES:
    r = subprocess.run(['node', f'{m}/util/construir.js'], capture_output=True, text=True)
    print(m, '→', 'bundle listo' if r.returncode == 0 else r.stdout + r.stderr)
!nvidia-smi --query-gpu=index,name,memory.total,memory.free,clocks.max.sm --format=csv
!lscpu | grep -E '^(Model name|CPU\(s\))'
for m in MOTORES:
    print(f'\n{m}:')
    print(subprocess.run(['node', '-e', f"require('./{m}/bundle.js'); const x = globalThis.EspanolLike.crearMotor({{}}); console.log(x.vm.c1en.disponibles().join('\\n'))"],
                         capture_output=True, text=True, env=dict(os.environ, C1EN_GPU=str(GPU))).stdout)

## Primero: ¿NEÑ da lo mismo en la T4 que en la CPU?

La suite de la fase N2 en la GPU de verdad (`--gpu`): 29 operaciones de NEÑ en jit y en la T4, comparadas al bit (seno y exp a 1e-14); el tráfico por el PCIe de tres cadenas (los datos tienen que quedarse en la GPU); y `fourier`, `por`, `x*2+1`… contra el motor LTS, al bit. Si algo sale ✗, lo que se mida después no vale: avísame con la salida.

In [ ]:
# el LTS hace falta para la comparación «iguales al LTS»
if 'motor' not in MOTORES and os.path.exists('motor/util/construir.js'):
    subprocess.run(['node', 'motor/util/construir.js'], capture_output=True)
if 'motor-exp' in MOTORES and os.path.exists('motor-exp/pruebas/exp-nen-gpu.js'):
    r = subprocess.run(['node', 'motor-exp/pruebas/exp-nen-gpu.js', '--gpu'], capture_output=True, text=True,
                       env=dict(os.environ, C1EN_GPU=str(GPU)))
    print(r.stdout[-3000:], r.stderr[-2000:])
else:
    print('esta versión del proyecto no trae motor-exp/pruebas/exp-nen-gpu.js (fase N2)')

## Las pruebas de Ñ en cada dispositivo

Un proceso por motor y dispositivo. Cada prueba se corre una vez para calentar (ahí se compila el núcleo con gcc o NVRTC: sale aparte, en la columna `primera_ms`) y luego se queda la mejor de `REPETICIONES`. El log de cada corrida queda en `/kaggle/working/velocidad/`.

In [ ]:
import pandas as pd
os.makedirs(f'{TRABAJO}/velocidad', exist_ok=True)
COLS = ['familia', 'prueba', 'tam', 'dispositivo', 'ms', 'unidades', 'huella', 'primera_ms', 'veces', 'nota']
filas = []
for m in MOTORES:
    for d in DISPOSITIVOS:
        log = f'{TRABAJO}/velocidad/{m}-{d}.log'
        t0 = time.time()
        env = dict(os.environ, C1EN_GPU=str(GPU))
        sal = subprocess.run(['node', f'{m}/util/n-node.js', 'c1en/velocidad-c1en.esl', d, ESCALA, str(REPETICIONES), str(PRESUPUESTO_MS)],
                             capture_output=True, text=True, env=env)
        open(log, 'w').write(sal.stdout + sal.stderr)
        cabeza = next((l for l in sal.stdout.splitlines() if l.startswith('# dispositivo')), '')
        print(f'{m:10s} {d:7s} {time.time() - t0:6.0f} s · {cabeza[2:]}' + ('' if sal.returncode == 0 else f'  ✗ salió con {sal.returncode}'))
        if sal.returncode: print('   ', '\n    '.join((sal.stdout + sal.stderr).splitlines()[-8:]))
        for l in sal.stdout.splitlines():
            if l.startswith('RES|'):
                f = dict(zip(COLS, l.split('|')[1:]))
                f['motor'] = m
                filas.append(f)
df = pd.DataFrame(filas)
for c in ['ms', 'unidades', 'huella', 'primera_ms', 'veces']:
    df[c] = pd.to_numeric(df[c], errors='coerce')
df['por_s'] = df['unidades'] / (df['ms'] / 1000)          # elementos, flops o iteraciones por segundo
print(f'\n{len(df)} medidas · {int((df["nota"] == "saltada").sum())} saltadas por el presupuesto')

## ECOSIM-G v3.1 entero: ms por tick

El programa grande del proyecto, con 2.000 plazas por mundo; el `jit` solo con 20 mundos porque es lento. Primero una corrida de calentamiento (compila los núcleos); después cada caso se corre con 60 y con 600 ticks y se resta. Lo que importa es **ms por tick y por mundo**: con eso se calcula cuánto tarda una corrida de miles de mundos (la tabla del cuaderno `kaggle-ecosim-g3.ipynb`).

In [ ]:
ECOSIM = {'jit': [20], 'nativo': [20, 100], 'gpu': [100, 1000]}
TICKS_ECO = 600
# Los primeros ticks incluyen compilar los núcleos (gcc o NVRTC). Para no
# contarlo, cada caso se corre dos veces, con TICKS_ECO y con TICKS_ECO / 10, y
# el tiempo por tick es la diferencia: (T(600) − T(60)) / 540.
def ecosim(m, d, mundos, ticks):
    env = dict(os.environ, C1EN_DISPOSITIVO=d, C1EN_GPU=str(GPU))
    sal = subprocess.run(['node', f'{m}/util/n-node.js', '--c1en', 'ecosim/ecosim-g3.esl', str(mundos), str(ticks), '2000',
                          '20261018', f'{TRABAJO}/velocidad/serie-{m}-{d}-{mundos}-{ticks}.csv', '6', '1', '0',
                          f'{TRABAJO}/velocidad/eco-{m}-{d}-{mundos}-{ticks}.csv'],
                         capture_output=True, text=True, env=env)
    open(f'{TRABAJO}/velocidad/eco-{m}-{d}-{mundos}-{ticks}.log', 'w').write(sal.stdout + sal.stderr)
    r = re.search(r'en ([\d.]+) ms ·', sal.stdout)
    if not r: print(f'{m} {d} {mundos} × {ticks}: no salió la medida'); print('\n'.join((sal.stdout + sal.stderr).splitlines()[-6:]))
    return (float(r.group(1)) if r else None), ('invariantes ✓' in sal.stdout)
eco = []
if os.path.exists('ecosim/ecosim-g3.esl'):
    for m in MOTORES:
        for d in DISPOSITIVOS:
            for mundos in ECOSIM.get(d, []):
                t0 = time.time()
                # calentar: la primera corrida compila los núcleos y los deja en la caché del
                # disco; si no, la de 60 ticks pagaría la compilación y la resta saldría mal
                # (en la primera medida de Kaggle, nativo con 20 mundos dio 0 ms por tick)
                ecosim(m, d, 2, 2)
                corto, ok1 = ecosim(m, d, mundos, TICKS_ECO // 10)
                largo, ok2 = ecosim(m, d, mundos, TICKS_ECO)
                if corto is None or largo is None: continue
                ms_tick = max(largo - corto, 1e-9) / (TICKS_ECO - TICKS_ECO // 10)
                eco.append(dict(motor=m, familia='ecosim', prueba='ecosim-g3', tam=str(mundos), dispositivo=d, ms=ms_tick,
                                unidades=mundos, ms_mundo=ms_tick / mundos, primera_ms=corto, invariantes=ok1 and ok2, pared_s=time.time() - t0))
                print(f'{m:10s} {d:7s} {mundos:5d} mundos: {ms_tick:9.2f} ms/tick · {ms_tick / mundos:7.4f} ms/tick y mundo · '
                      f'invariantes {"✓" if ok1 and ok2 else "✗"} · {time.time() - t0:.0f} s de pared')
eco = pd.DataFrame(eco)
if len(eco):
    # cuánto tardarían 12.000 mundos × 6 años en UNA T4, con lo medido
    g = eco[eco.dispositivo == 'gpu'].sort_values('unidades').tail(1)
    if len(g):
        msm = float(g.ms_mundo.iloc[0])
        print(f'\ncon {msm:.4f} ms por tick y mundo: 12.000 mundos × 21.600 ticks ≈ {msm * 12000 * 21600 / 1000 / 60:.0f} min en una T4 '
              f'(≈ la mitad con las dos)')

## NumPy y CuPy haciendo lo mismo

La referencia de Python, con los mismos tamaños. **CuPy** se mide de dos maneras: con los datos ya en la GPU (fila `senoA`, lo mejor que puede hacer la T4) y subiendo y bajando los datos (fila `senoA+copia`, `expA+copia`…). Desde motor-exp N2, NEÑ en `gpu` deja los arreglos en la GPU entre operaciones igual que CuPy, así que cada fila se compara con su par.

El π de NumPy/CuPy usa su propio generador de azar (no `azarHilo`), así que su huella no se compara.

In [ ]:
import numpy as np
try:
    import cupy as cp
except ImportError:
    subprocess.run('pip install -q cupy-cuda12x', shell=True)
    try: import cupy as cp
    except ImportError: cp = None
print('NumPy', np.__version__, '· CuPy', cp.__version__ if cp else 'no está')
if cp: cp.cuda.Device(GPU).use()

def mejor(f, rep=REPETICIONES, sincronizar=False):
    f()
    if sincronizar: cp.cuda.Stream.null.synchronize()
    t = 1e300
    for _ in range(rep):
        t0 = time.perf_counter(); r = f()
        if sincronizar: cp.cuda.Stream.null.synchronize()
        t = min(t, (time.perf_counter() - t0) * 1000)
    return t, r

TAM = {'chica':  dict(saxpy=[1 << 16], pi=[4096], mandel=[200], cuerpos=[1024], histo=[1 << 16], nen=[1 << 16], filas=[256], por=[128, 256], fft=[1 << 14], vm=[1 << 16]),
       'normal': dict(saxpy=[1 << 20, 1 << 24], pi=[65536, 1 << 20], mandel=[1200, 4000], cuerpos=[8192, 32768], histo=[1 << 22],
                      nen=[1 << 20, 1 << 24], filas=[4096], por=[256, 512, 1024, 2048], fft=[1 << 20, 1 << 22], vm=[1 << 22]),
       'grande': dict(saxpy=[1 << 20, 1 << 24, 40_000_000], pi=[1 << 20, 1 << 22], mandel=[4000, 8000], cuerpos=[32768, 65536], histo=[1 << 24],
                      nen=[1 << 24, 40_000_000], filas=[16384], por=[1024, 2048, 4096], fft=[1 << 22, 1 << 24], vm=[1 << 24])}[ESCALA]

def mandel_xp(xp, ancho, alto, it=500):
    cx = (-2.0 + 3.0 * (xp.arange(ancho * alto) % ancho) / ancho)
    cy = (-1.2 + 2.4 * (xp.arange(ancho * alto) // ancho) / alto)
    zx = xp.zeros_like(cx); zy = xp.zeros_like(cx); k = xp.zeros(cx.shape, dtype=xp.int32)
    for _ in range(it):
        vivo = zx * zx + zy * zy < 4.0
        t = zx * zx - zy * zy + cx
        zy = xp.where(vivo, 2.0 * zx * zy + cy, zy); zx = xp.where(vivo, t, zx); k += vivo
    return k

def cuerpos_xp(xp, x, y, trozo=4096):
    ax = xp.empty_like(x); ay = xp.empty_like(y)
    for a in range(0, len(x), trozo):            # por trozos: 32 768² no cabe de una vez
        dx = x[None, :] - x[a:a + trozo, None]; dy = y[None, :] - y[a:a + trozo, None]
        d2 = dx * dx + dy * dy + 0.01; inv = 1.0 / (d2 * xp.sqrt(d2))
        ax[a:a + trozo] = (dx * inv).sum(1); ay[a:a + trozo] = (dy * inv).sum(1)
    return ax, ay

def softmax_xp(xp, m):
    e = xp.exp(m - m.max(1, keepdims=True)); return e / e.sum(1, keepdims=True)

ref = []
def anotar(lib, familia, prueba, tam, unidades, t, huella):
    ref.append(dict(motor=lib, dispositivo=lib, familia=familia, prueba=prueba, tam=str(tam), ms=t, unidades=unidades, huella=float(huella)))

PI_MUESTRAS = 1000
for lib, xp, sinc in [('numpy', np, False)] + ([('cupy', cp, True)] if cp else []):
    rng = xp.random.default_rng(1) if lib == 'numpy' else xp.random.default_rng(1)
    for n in TAM['saxpy']:
        x = rng.random(n); y = rng.random(n)
        t, z = mejor(lambda: 2.5 * x + y, sincronizar=sinc); anotar(lib, 'nucleos', 'saxpy', n, 2.0 * n, t, z.sum())
    for h in TAM['pi']:
        def pi():
            c = 0
            for a in range(0, h * PI_MUESTRAS, 1 << 24):
                k = min(1 << 24, h * PI_MUESTRAS - a); p = rng.random((2, k)); c += int(((p * p).sum(0) < 1.0).sum())
            return c
        t, c = mejor(pi, rep=1, sincronizar=sinc); anotar(lib, 'nucleos', 'pi', f'{h}x{PI_MUESTRAS}', 1.0 * h * PI_MUESTRAS, t, c)
    for lado in TAM['mandel']:
        if lib == 'numpy' and lado > 1200: continue        # 500 pasadas sobre 10 millones de puntos: minutos en NumPy
        t, k = mejor(lambda: mandel_xp(xp, lado, lado * 2 // 3), rep=1, sincronizar=sinc)
        anotar(lib, 'nucleos', 'mandelbrot', f'{lado}x{lado * 2 // 3}', 1.0 * lado * (lado * 2 // 3), t, k.sum())
    for nc in TAM['cuerpos']:
        if lib == 'numpy' and nc > 8192: continue
        x = rng.random(nc) * 10.0; y = rng.random(nc) * 10.0
        t, (ax, ay) = mejor(lambda: cuerpos_xp(xp, x, y), rep=1, sincronizar=sinc); anotar(lib, 'nucleos', 'n-cuerpos', nc, 20.0 * nc * nc, t, ax.sum() + ay.sum())
    for n in TAM['histo']:
        v = rng.random(n)
        t, h = mejor(lambda: xp.bincount(xp.minimum((v * 256).astype(xp.int32), 255), minlength=256), sincronizar=sinc)
        anotar(lib, 'nucleos', 'histograma', n, 1.0 * n, t, h.sum())
    # las de NEÑ: con los datos ya en el dispositivo, y (CuPy) con la copia de ida y vuelta
    # «cupy+copia» se anota como dispositivo cupy y prueba «…+copia», para ir en la
    # misma fila que «senoA+copia» de Ñ (que sube y baja los datos igual)
    variantes = [(lib, False)] + ([('cupy+copia', True)] if lib == 'cupy' else [])
    for nombre, copia in variantes:
        sube = (lambda a: cp.asarray(a)) if copia else (lambda a: a)
        baja = (lambda a: cp.asnumpy(a)) if copia else (lambda a: a)
        base = np if copia else xp
        for n in TAM['nen']:
            x = base.linspace(0.0, 1.0, n)
            t, r = mejor(lambda: baja(xp.sin(sube(x))), sincronizar=sinc); anotar(nombre, 'nen', 'senoA', n, 1.0 * n, t, r.sum())
            t, r = mejor(lambda: baja(xp.exp(sube(x))), sincronizar=sinc); anotar(nombre, 'nen', 'expA', n, 1.0 * n, t, r.sum())
            if not copia:
                t, r = mejor(lambda: float((xp.exp(xp.sin(x)) * 2 + 1).sum()), sincronizar=sinc); anotar(nombre, 'nen', 'cadena', n, 4.0 * n, t, r)
        for f in TAM['filas']:
            m = (np.random.default_rng(2).random((f, 1024)) if copia else rng.random((f, 1024)))
            t, r = mejor(lambda: baja(softmax_xp(xp, sube(m))), sincronizar=sinc); anotar(nombre, 'nen', 'softmax', f'{f}x1024', 1.0 * f * 1024, t, r.sum())
            t, r = mejor(lambda: baja(sube(m).sum(1)), sincronizar=sinc); anotar(nombre, 'nen', 'sumaEje', f'{f}x1024', 1.0 * f * 1024, t, r.sum())
        for lado in TAM['por']:
            A = (np.random.default_rng(3).random((lado, lado)) if copia else rng.random((lado, lado))); B = A.T.copy()
            t, r = mejor(lambda: baja(sube(A) @ sube(B)), sincronizar=sinc); anotar(nombre, 'nen', 'por', f'{lado}x{lado}', 2.0 * lado ** 3, t, r.sum())
        for n in TAM['fft']:
            x = base.linspace(0.0, 1.0, n)
            t, r = mejor(lambda: baja(xp.fft.fft(sube(x)).real), sincronizar=sinc); anotar(nombre, 'nen', 'fourier', n, 5.0 * n * np.log2(n), t, (r * r).sum())
    for n in TAM['vm']:
        x = xp.linspace(0.0, 1.0, n)
        t, r = mejor(lambda: x * 2 + 1, sincronizar=sinc); anotar(lib, 'arit', 'x*2+1', n, 2.0 * n, t, r.sum())
        t, r = mejor(lambda: x.sum(), sincronizar=sinc); anotar(lib, 'arit', 'suma', n, 1.0 * n, t, r)
    print(lib, 'listo')
ref = pd.DataFrame(ref)
cc = ref['motor'] == 'cupy+copia'
ref.loc[cc, 'prueba'] = ref.loc[cc, 'prueba'] + '+copia'
ref.loc[cc, ['motor', 'dispositivo']] = 'cupy'
ref['por_s'] = ref['unidades'] / (ref['ms'] / 1000)

## La tabla

Milisegundos (la mejor de las repeticiones; menos es mejor; en `ecosim`, ms por tick de todos los mundos). Las columnas `gpu × jit` y `gpu × nativo` son cuántas veces más rápida es la GPU; con dos motores, `motor-exp × motor` es cuántas veces más rápido es el experimental que el LTS. `vs cupy` es cuántas veces más lenta es la GPU de Ñ que CuPy en la misma T4 (1,0 = igual).

In [ ]:
todo = pd.concat([df, ref] + ([eco] if len(eco) else []), ignore_index=True)
todo['col'] = todo.apply(lambda r: r['dispositivo'] if r['motor'] in ('numpy', 'cupy') or len(MOTORES) == 1 else f"{r['dispositivo']} ({r['motor']})", axis=1)
orden_tam = lambda s: s.map(lambda t: float(str(t).split('x')[0]) if str(t).split('x')[0].replace('.', '').isdigit() else 0)
ancho = todo.pivot_table(index=['familia', 'prueba', 'tam'], columns='col', values='ms', aggfunc='min')
ancho = ancho.reset_index().sort_values(['familia', 'prueba', 'tam'], key=lambda s: orden_tam(s) if s.name == 'tam' else s).set_index(['familia', 'prueba', 'tam'])
col_de = lambda d, m=MOTORES[0]: d if len(MOTORES) == 1 else f'{d} ({m})'
g = col_de('gpu') if col_de('gpu') in ancho else None
if len(MOTORES) > 1:       # cuántas veces más rápido es el primer motor que el otro, en cada dispositivo
    for d in DISPOSITIVOS:
        a, b = col_de(d, MOTORES[0]), col_de(d, MOTORES[1])
        if a in ancho and b in ancho: ancho[f'{d}: {MOTORES[0]} × {MOTORES[1]}'] = (ancho[b] / ancho[a]).round(1)
if g:
    for base in ['jit', 'nativo']:
        b = col_de(base)
        if b in ancho: ancho[f'gpu × {base}'] = (ancho[b] / ancho[g]).round(1)
    if 'cupy' in ancho: ancho['vs cupy'] = (ancho[g] / ancho['cupy']).round(1)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 30)
print(ancho.round(2).to_string())

## ¿Dan lo mismo? La huella en los tres dispositivos

C1EÑ promete los mismos bits en `jit`, `nativo` y `gpu` para `+ − * /` y `raiz`. Las pruebas sin funciones trascendentes (saxpy, π, mandelbrot, n-cuerpos, histograma, `por`, `sumaEje`, `fourier` —sus tablas de senos se calculan en la CPU—, `x*2+1`, `suma`) tienen que dar **la misma huella exacta**. `senoA`, `expA`, `cadena` y `softmax` llaman a seno o exponencial: ahí la GPU puede diferir en pocos ulp y la huella en la última cifra.

In [ ]:
nucleo = df[df['nota'] != 'saltada']
TRASC = {'senoA', 'expA', 'senoA+copia', 'cadena', 'softmax'}
malos = 0
for (m, fam, p, tam), g in nucleo.groupby(['motor', 'familia', 'prueba', 'tam']):
    h = dict(zip(g['dispositivo'], g['huella']))
    if 'jit' not in h or len(h) < 2: continue
    for d, v in h.items():
        if d == 'jit': continue
        rel = abs(v - h['jit']) / max(abs(h['jit']), 1e-300)
        if p in TRASC: estado = 'igual' if v == h['jit'] else f'≈ ({rel:.1e})'
        else: estado = 'igual' if v == h['jit'] else f'DISTINTO ({rel:.1e})'
        if estado.startswith('DISTINTO'): malos += 1
        if estado != 'igual' or p not in TRASC: print(f'{m:10s} {p:11s} {tam:>12s}  {d:7s} {estado}')
print('\n' + ('✓ todas las pruebas exactas dan la misma huella en los tres dispositivos' if not malos else f'✗ {malos} pruebas exactas NO dan lo mismo: mira arriba'))

## Las gráficas

In [ ]:
import matplotlib.pyplot as plt
COLORES = {'jit': '#c0392b', 'nativo': '#d68910', 'gpu': '#1e8449', 'numpy': '#5d6d7e', 'cupy': '#2e86c1', 'cupy+copia': '#85c1e9'}
import matplotlib.colors as mc
def color(c):
    base = mc.to_rgb(COLORES.get(c.split(' ')[0], '#7f8c8d'))
    # con dos motores, el segundo (normalmente el LTS) sale del mismo color, más claro
    return tuple(x + (1 - x) * 0.55 for x in base) if len(MOTORES) > 1 and c.endswith(f'({MOTORES[1]})') else base
estilo = lambda c: '--' if len(MOTORES) > 1 and c.endswith(f'({MOTORES[1]})') else '-'
# 1 · el tamaño más grande de cada prueba, en ms (escala log)
grande = todo[todo['ms'].notna()].copy()
grande['n'] = orden_tam(grande['tam'])
mayor = grande.groupby(['prueba'])['n'].transform('max')
g1 = grande[grande['n'] == mayor].pivot_table(index='prueba', columns='col', values='ms', aggfunc='min')
cols = [c for c in ['jit', 'nativo', 'gpu', 'numpy', 'cupy', 'cupy+copia'] if c in g1] + [c for c in g1 if c not in COLORES]
ax = g1[cols].plot.bar(figsize=(15, 6), logy=True, color=[color(c) for c in cols], width=0.8)
ax.legend(title=None, ncol=2, fontsize=8); ax.set_ylabel('ms (menos es mejor, escala log)'); ax.set_title(f'C1EÑ: el tamaño más grande de cada prueba · escala {ESCALA}')
plt.xticks(rotation=30, ha='right'); plt.tight_layout(); plt.show()

# 2 · rendimiento según el tamaño: matrices (GFLOP/s) y saxpy (GB/s aprox.)
fig, ax = plt.subplots(1, 3, figsize=(17, 5))
for a, (p, esc, nombre) in zip(ax, [('por', 1e9, 'GFLOP/s · por (matrices)'), ('saxpy', 1e9, 'G operaciones/s · saxpy'), ('senoA', 1e6, 'M elementos/s · senoA')]):
    s = grande[grande['prueba'] == p]
    for c, gg in s.groupby('col'):
        gg = gg.sort_values('n'); a.plot(gg['n'], gg['por_s'] / esc, marker='o', linestyle=estilo(c), label=c, color=color(c))
    a.set_xscale('log'); a.set_yscale('log'); a.set_title(nombre); a.set_xlabel('tamaño'); a.legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# Los CSV para descargar (panel Output) — o para mandármelos y ajustar el motor con datos de verdad
todo.drop(columns=['col']).to_csv(f'{TRABAJO}/c1en-velocidad.csv', index=False)
ancho.to_csv(f'{TRABAJO}/c1en-velocidad-resumen.csv')
info = dict(escala=ESCALA, motores=MOTORES, repeticiones=REPETICIONES, presupuesto_ms=PRESUPUESTO_MS,
            gpu=(subprocess.run(['nvidia-smi', '--query-gpu=name,driver_version', '--format=csv,noheader', f'--id={GPU}'], capture_output=True, text=True).stdout.strip()
                 if shutil.which('nvidia-smi') else 'sin nvidia-smi'),
            commit=subprocess.run(['git', 'log', '-1', '--format=%h'], capture_output=True, text=True).stdout.strip())
json.dump(info, open(f'{TRABAJO}/c1en-velocidad-info.json', 'w'), ensure_ascii=False, indent=1)
for f in ['c1en-velocidad.csv', 'c1en-velocidad-resumen.csv', 'c1en-velocidad-info.json']:
    print(f'{TRABAJO}/{f}: {os.path.getsize(f"{TRABAJO}/{f}") / 1024:.1f} KB')
print(info)

## Cómo leerlo

- **nucleos en `gpu` frente a `jit`**: es lo que C1EÑ promete. Un núcleo con mucho cálculo por elemento (mandelbrot, n-cuerpos, π) debería ir cientos de veces más rápido en la T4; saxpy, que casi no calcula, lo limita la memoria y el lanzamiento.
- **nen en `gpu`** (motor-exp N2): los arreglos se quedan en la GPU entre operaciones, así que `senoA` se compara con `cupy` (datos ya allí) y `senoA+copia` con su fila de CuPy con copia (subir y bajar por el PCIe, ~6–12 GB/s en Kaggle). `cadena` dice cuánto cuesta una expresión entera sin bajar nada a mitad. En `por` el cálculo crece como n³; C1EÑ no usa FMA (para dar los mismos bits que la CPU), así que su techo en la T4 es la mitad del de cuBLAS.
- **vs cupy**: cuánto le falta a C1EÑ para lo que da la misma T4 con bibliotecas de NVIDIA (cuBLAS en `@`, cuFFT en `fft`). En `por` y `fourier` CuPy usa cuBLAS y cuFFT, años de ajuste: la distancia dice cuánto margen hay en los núcleos de NEÑ.
- **ecosim**: `ms por tick y mundo` en la GPU con 1.000 mundos es el número para planear corridas grandes.
- Si `gpu` no aparece, mira la celda 3: `disponibles()` dice por qué (sin libnvrtc, driver…).

Los CSV quedan en `/kaggle/working` (*Output* → descargar). Mándamelos: con los tiempos de verdad se decide qué núcleo de NEÑ vale la pena afinar primero.